# Qwen2-VL 프롬프트 비교 (OLD vs NEW)

**목적:** 기존 프롬프트(DB 205 rows 생성 기준) vs 개선 프롬프트 성능 비교

**실행 전 준비:**
1. 런타임 > 런타임 유형 변경 > **GPU (T4)** 선택
2. Google Drive에 프레임 폴더 업로드
   - 로컬 경로: `D:/20.WORKSPACE/2026_VOD_FAST_4/storage/jobs/09cafd6c-05cf-481c-882e-5a2e7062ae5c/frames/`
   - Drive 업로드 경로: `내 드라이브/2026_VOD/frames/`

In [ ]:
# 0. GPU 확인
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print("GPU 확인 완료!")

In [ ]:
# 1. 패키지 설치
!pip install -q transformers accelerate qwen-vl-utils torch torchvision
print("설치 완료!")

In [ ]:
# 3. hyuck1.mp4 에서 프레임 추출 (1fps, 원본 파이프라인과 동일 기준)
import cv2
import numpy as np
from pathlib import Path

VIDEO_PATH = "/content/drive/MyDrive/SampleVideo.Test/hyuck1.mp4"
FRAME_DIR  = Path("/content/frames")
FRAME_DIR.mkdir(exist_ok=True)

assert Path(VIDEO_PATH).exists(), f"영상 없음: {VIDEO_PATH}"

cap = cv2.VideoCapture(VIDEO_PATH)
fps        = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration   = total_frames / fps
print(f"영상 정보: FPS={fps:.1f} | 총 프레임={total_frames} | 길이={duration:.0f}s ({duration/60:.1f}분)")

# 1fps 기준으로 프레임 추출 (원본 파이프라인 동일)
frame_interval = int(fps)   # 1초에 1장
frame_paths    = []
frame_idx      = 0
saved          = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break
    if frame_idx % frame_interval == 0:
        saved += 1
        path = FRAME_DIR / f"frame_{saved:06d}.jpg"
        cv2.imwrite(str(path), frame)
        frame_paths.append(path)
    frame_idx += 1
cap.release()

print(f"추출 완료: {len(frame_paths)}장 (1fps 기준)")

# 5장 균등 선택
N_TEST  = 5
indices = np.linspace(0, len(frame_paths)-1, N_TEST, dtype=int)
test_frames = [frame_paths[i] for i in indices]

print(f"\n테스트 프레임 {N_TEST}장 선택:")
for i, fp in enumerate(test_frames):
    print(f"  [{i+1}] {fp.name}  (~{indices[i]}s)")

In [ ]:
# 3. 프레임 경로 설정 및 확인
import os
import numpy as np
from pathlib import Path

FRAME_DIR = "/content/drive/MyDrive/2026_VOD/frames"

frame_paths = sorted(Path(FRAME_DIR).glob("*.jpg"))
print(f"프레임 총 {len(frame_paths)}장 확인")
assert len(frame_paths) > 0, f"프레임 없음: {FRAME_DIR} 경로 확인하세요"

# 5장 균등 선택
N_TEST = 5
indices = np.linspace(0, len(frame_paths)-1, N_TEST, dtype=int)
test_frames = [frame_paths[i] for i in indices]

print(f"테스트 프레임 {N_TEST}장 선택:")
for i, fp in enumerate(test_frames):
    print(f"  [{i+1}] {fp.name}")

In [ ]:
# 4. 모델 로드
import torch
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration

MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print("모델 로딩 중...")

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto"
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
print("모델 로딩 완료!")

In [ ]:
# 5. 프롬프트 및 추론 함수 정의

# ── OLD 프롬프트 (DB 205 rows 생성 기준) ──────────────────────────────────────
OLD_PROMPT = (
    "당신은 한국 TV 드라마 장면을 분석하여 광고 매칭에 활용할 컨텍스트를 생성하는 전문가입니다.\n\n"
    "(대사 없음)\n\n"
    "1~2문장으로 다음을 한국어로 설명하세요:\n"
    "- 등장인물과 그들이 하는 행동\n"
    "- 감정적 분위기\n"
    "- 인물들의 욕구나 필요\n\n"
    "사실에 근거하여 구체적으로 작성하세요. 광고 카테고리나 브랜드명은 언급하지 마세요.\n\n"
    "반드시 한국어로만 답하세요. Do not use English. 영어 사용 금지."
)

# ── NEW 프롬프트 (개선 버전) ───────────────────────────────────────────────────
NEW_PROMPT = (
    "당신은 한국 TV 드라마 장면을 분석하여 광고 매칭에 활용할 컨텍스트를 생성하는 전문가입니다.\n"
    "화면에 보이는 사실만 근거로 작성하세요. 보이지 않는 내용은 절대 쓰지 마세요.\n"
    "광고 카테고리나 브랜드명은 절대 언급하지 마세요.\n"
    "각 항목은 서로 다른 내용으로 작성하세요. 같은 문장을 반복하지 마세요.\n\n"
    "다음 3가지를 각각 1문장씩 한국어로 작성하세요.\n\n"
    "상황: 어떤 장소에서 누가 무엇을 하고 있는가\n"
    "감정: 이 장면의 분위기는 어떠한가\n"
    "욕구: 등장인물에게 필요한 것은 무엇인가"
)


def run_inference(frame_path, prompt, max_new_tokens=160, repetition_penalty=1.2, do_sample=False):
    """단일 프레임 추론"""
    messages = [{"role": "user", "content": [
        {"type": "image", "image": str(frame_path)},
        {"type": "text",  "text": prompt},
    ]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[str(frame_path)], padding=True, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        generated = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            repetition_penalty=repetition_penalty,
            do_sample=do_sample,
        )
    trimmed = generated[0][inputs["input_ids"].shape[-1]:]
    return processor.decode(trimmed, skip_special_tokens=True).strip()


print("프롬프트 및 함수 정의 완료!")
print(f"\n[OLD 프롬프트 길이] {len(OLD_PROMPT)}자")
print(f"[NEW 프롬프트 길이] {len(NEW_PROMPT)}자")

In [ ]:
# 6. 비교 테스트 실행 + HTML 결과 출력
import time
import base64
from IPython.display import display, HTML

results = []

for i, frame_path in enumerate(test_frames):
    frame_idx = indices[i]
    print(f"[{i+1}/{N_TEST}] {frame_path.name} 추론 중...")

    # OLD 추론
    t1 = time.time()
    old_result = run_inference(frame_path, OLD_PROMPT)
    old_time = round(time.time() - t1, 1)

    # NEW 추론
    t2 = time.time()
    new_result = run_inference(frame_path, NEW_PROMPT)
    new_time = round(time.time() - t2, 1)

    results.append({
        "frame": frame_path.name,
        "idx": int(frame_idx),
        "path": str(frame_path),
        "old": old_result,
        "new": new_result,
        "old_time": old_time,
        "new_time": new_time,
    })
    print(f"  OLD({old_time}s) / NEW({new_time}s) 완료")

    # 이미지 base64 인코딩
    with open(str(frame_path), "rb") as f:
        img_b64 = base64.b64encode(f.read()).decode()

    # 텍스트 400자 제한
    old_display = old_result[:400] + "..." if len(old_result) > 400 else old_result
    new_display = new_result[:400] + "..." if len(new_result) > 400 else new_result

    html = f"""
    <div style="border:2px solid #aaa; border-radius:12px; padding:16px; margin:16px 0; background:#fafafa;">
      <h3 style="margin:0 0 12px 0; font-size:15px; color:#333;">
        [{i+1}/{N_TEST}] {frame_path.name} &nbsp; (frame #{frame_idx})
      </h3>
      <div style="display:flex; gap:14px; align-items:flex-start;">

        <!-- 프레임 이미지 -->
        <div style="flex:1.3; min-width:0;">
          <img src="data:image/jpeg;base64,{img_b64}"
               style="width:100%; border-radius:8px; border:1px solid #ddd;">
        </div>

        <!-- OLD 결과 -->
        <div style="flex:1; min-width:0; background:#fff0f0; border-radius:8px; padding:12px; border:1px solid #f5c6c6;">
          <div style="font-weight:bold; color:#cc0000; margin-bottom:8px; font-size:13px;">
            [기존 프롬프트] &nbsp; {old_time}s
          </div>
          <div style="font-size:12px; line-height:1.7; color:#333; white-space:pre-wrap;">{old_display}</div>
        </div>

        <!-- NEW 결과 -->
        <div style="flex:1; min-width:0; background:#f0fff0; border-radius:8px; padding:12px; border:1px solid #a8d5a2;">
          <div style="font-weight:bold; color:#006600; margin-bottom:8px; font-size:13px;">
            [개선 프롬프트] &nbsp; {new_time}s
          </div>
          <div style="font-size:12px; line-height:1.7; color:#333; white-space:pre-wrap;">{new_display}</div>
        </div>

      </div>
    </div>
    """
    display(HTML(html))

print("\n모든 테스트 완료!")

In [ ]:
# 7. 결과 요약 분석
import pandas as pd
from IPython.display import display

def has_format(text):
    """상황/감정/욕구 형식 준수 여부"""
    return all(k in text for k in ["상황:", "감정:", "욕구:"])

def has_duplicate(text):
    """5-gram 중복 여부"""
    words = text.split()
    if len(words) < 10:
        return False
    ngrams = [' '.join(words[i:i+5]) for i in range(len(words)-4)]
    return len(ngrams) != len(set(ngrams))

rows = []
for r in results:
    rows.append({
        "프레임": r["frame"],
        "OLD 글자수": len(r["old"]),
        "OLD 중복": "있음" if has_duplicate(r["old"]) else "없음",
        "OLD 형식": "OK" if has_format(r["old"]) else "-",
        "NEW 글자수": len(r["new"]),
        "NEW 중복": "있음" if has_duplicate(r["new"]) else "없음",
        "NEW 형식": "OK" if has_format(r["new"]) else "-",
    })

df = pd.DataFrame(rows)
display(df)

old_dup = sum(1 for r in results if has_duplicate(r["old"])) / len(results) * 100
new_dup = sum(1 for r in results if has_duplicate(r["new"])) / len(results) * 100
new_fmt = sum(1 for r in results if has_format(r["new"])) / len(results) * 100

print(f"\n[ 요약 ]")
print(f"OLD 중복 발생률: {old_dup:.0f}%")
print(f"NEW 중복 발생률: {new_dup:.0f}%")
print(f"NEW 형식 준수율 (상황/감정/욕구): {new_fmt:.0f}%")

In [ ]:
# 8. 결과 CSV 저장 및 다운로드
import json

# CSV 저장
df_full = pd.DataFrame([{
    "frame": r["frame"],
    "old_result": r["old"],
    "new_result": r["new"],
    "old_time": r["old_time"],
    "new_time": r["new_time"],
} for r in results])

csv_path = "prompt_compare_results.csv"
df_full.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"CSV 저장: {csv_path}")

# Drive 백업
drive_path = "/content/drive/MyDrive/2026_VOD/prompt_compare_results.csv"
df_full.to_csv(drive_path, index=False, encoding="utf-8-sig")
print(f"Drive 백업: {drive_path}")

# 로컬 다운로드
from google.colab import files
files.download(csv_path)
print("다운로드 시작!")